In [2]:
import pandas as pd
import ast
import numpy as np
import os
import pronto
from itertools import islice
from tqdm import tqdm
import json
from transformers import AutoTokenizer, AutoModel


/opt/anaconda3/envs/preclinical/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Entity normalization tests

### Load MONDO terminology
Note: 
- https://mondo.monarchinitiative.org/pages/download/

In [6]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)


In [8]:
%%capture

# Load the ontology
ontology = pronto.Ontology("data/mondo/mondo.owl")


In [9]:
len(ontology.terms())

56452

In [10]:
mondo_terms = [t for t in ontology.terms() if t.id.startswith("MONDO:")]
print(f"Total MONDO terms: {len(mondo_terms)}")

# Collect all term names and synonyms as a flat, unique set
all_names = set()

for term in mondo_terms:
    if term.name:
        all_names.add(term.name.strip())
    for synonym in term.synonyms:
        all_names.add(synonym.description.strip())

# Convert to sorted list and save
print(f"Total MONDO terms with synonyms: {len(all_names)}")


Total MONDO terms: 30158
Total MONDO terms with synonyms: 129189


In [14]:
term = ontology["MONDO:0007147"]

# 🔍 Print the synonyms
print(f"Synonyms for {term.name} ({term.id}):")
for synonym in term.synonyms:
    print(f"- {synonym.description} ({synonym.scope})")

Synonyms for obstructive sleep apnea syndrome (MONDO:0007147):
- syndrome, upper airway resistance, sleep apnea (RELATED)
- Osa (RELATED)
- apnea, obstructive sleep (RELATED)
- sleep Apneas, obstructive (RELATED)
- obstructive sleep apnea (EXACT)
- obstructive sleep apnea syndrome (EXACT)
- syndrome, obstructive sleep apnea (RELATED)
- syndrome, sleep apnea, obstructive (RELATED)
- Apneas, obstructive sleep (RELATED)
- obstructive sleep Apneas (RELATED)
- sleep apnea hypopnea syndrome (RELATED)
- sleep apnea syndrome, obstructive (RELATED)
- upper airway resistance sleep apnea syndrome (RELATED)
- OSAHS (RELATED)
- sleep apnea/hypopnea syndrome (RELATED)


In [20]:
# 'neurotic depression', 'MONDO:0024614', ['relapsing-remitting multiple sclerosis', 'MONDO:0005314']
term = ontology["MONDO:0005314"]  # obsolete 46,XX sex reversal

# Get all superclasses (i.e., recursive parent terms)
print("All ancestor terms:")
for parent in term.superclasses():
    print(parent.id, "-", parent.name)

All ancestor terms:
MONDO:0005314 - relapsing-remitting multiple sclerosis
MONDO:0005301 - multiple sclerosis
MONDO:0000568 - autoimmune disorder of central nervous system
MONDO:0005560 - brain disorder
MONDO:0006704 - CNS demyelinating autoimmune disease
MONDO:0020800 - demyelinating disease of central nervous system
MONDO:0002602 - central nervous system disorder
MONDO:0002977 - autoimmune disorder of the nervous system
MONDO:0007179 - autoimmune disease
MONDO:0002562 - demyelinating disease
MONDO:0005071 - nervous system disorder
MONDO:0005046 - immune system disorder
MONDO:0005559 - neurodegenerative disease
MONDO:0021147 - disorder of development or morphogenesis
MONDO:0700096 - human disease
MONDO:0000001 - disease
BFO:0000016 - disposition
BFO:0000017 - realizable entity
BFO:0000020 - specifically dependent continuant
BFO:0000002 - continuant
BFO:0000001 - entity


In [28]:
term = ontology["MONDO:0000452"]  # obsolete 46,XX sex reversal

# Categorize synonyms by scope
scoped_synonyms = {
    "EXACT": [],
    "NARROW": [],
    "BROAD": [],
    "RELATED": []
}

for syn in term.synonyms:
    scope = syn.scope.upper()
    if scope in scoped_synonyms:
        syn_entry = syn.description
        if syn.xrefs:
            xrefs_str = ", ".join(str(xref) for xref in syn.xrefs)
            syn_entry += f" (xrefs: {xrefs_str})"
        scoped_synonyms[scope].append(syn_entry)

# Print nicely
for scope, synonyms in scoped_synonyms.items():
    print(f"\n{scope.capitalize()} synonyms:")
    for s in synonyms:
        print(f"- {s}")



Exact synonyms:
- progressive-relapsing MS (xrefs: Xref('DOID:0050785'))
- PRMS (xrefs: Xref('DOID:0050785'))

Narrow synonyms:

Broad synonyms:

Related synonyms:


In [46]:
import pronto
print(pronto.__version__)

2.7.0


In [52]:
from collections import deque
from functools import reduce

ont = ontology
ROOT_ID = "MONDO:0000001"                          # ‘disease or disorder’
ROOT = ont[ROOT_ID]

def _depth(term, root=ROOT):
    """
    Breadth-first search to compute the shortest ‘is_a’ path length
    between `term` and `root`, expressed as number of edges.

    Uses only `Term.superclasses(distance=1)` so it needs no extras.
    """
    if term == root:
        return 0
    visited = {term}
    queue   = deque([(term, 0)])                   # (node, depth_so_far)
    while queue:
        node, d = queue.popleft()
        for parent in node.superclasses(distance=1, with_self=False):
            if parent == root:
                return d + 1
            if parent not in visited:
                visited.add(parent)
                queue.append((parent, d + 1))
    # If ROOT is not reachable (shouldn’t happen in MONDO)
    return float("inf")

def lowest_common_ancestor(ids, root=ROOT):
    """
    Return the deepest (i.e. most specific) common ancestor
    of the MONDO terms in `ids`.
    """
    terms   = [ont[i] for i in ids]
    common  = reduce(set.intersection,
                     [set(t.superclasses(with_self=True)) for t in terms])
    if not common:
        raise ValueError("No common ancestor found")

    # Pick the ancestor with the *largest* depth (farthest from ROOT)
    return max(common, key=lambda t: _depth(t, root))

# Demo ------------------------------------------------------------------
lca = lowest_common_ancestor([
        "MONDO:0005314",   # relapsing-remitting MS
        "MONDO:0000452",   # progressive-relapsing MS
        "MONDO:0005301"    # multiple sclerosis
])
print(lca.id, lca.name)      # MONDO:0005301  multiple sclerosis

BFO:0000001 entity


In [30]:
#['progressive relapsing multiple sclerosis', 'MONDO:0000452']
term = ontology["MONDO:0000452"]  # obsolete 46,XX sex reversal

# Get all superclasses (i.e., recursive parent terms)
print("All ancestor terms:")
for parent in term.superclasses():
    print(parent.id, "-", parent.name)

📚 All ancestor terms:
MONDO:0000452 - progressive relapsing multiple sclerosis
MONDO:0005284 - chronic progressive multiple sclerosis
MONDO:0005301 - multiple sclerosis
MONDO:0000568 - autoimmune disorder of central nervous system
MONDO:0005560 - brain disorder
MONDO:0006704 - CNS demyelinating autoimmune disease
MONDO:0020800 - demyelinating disease of central nervous system
MONDO:0002602 - central nervous system disorder
MONDO:0002977 - autoimmune disorder of the nervous system
MONDO:0007179 - autoimmune disease
MONDO:0002562 - demyelinating disease
MONDO:0005071 - nervous system disorder
MONDO:0005046 - immune system disorder
MONDO:0005559 - neurodegenerative disease
MONDO:0021147 - disorder of development or morphogenesis
MONDO:0700096 - human disease
MONDO:0000001 - disease
BFO:0000016 - disposition
BFO:0000017 - realizable entity
BFO:0000020 - specifically dependent continuant
BFO:0000002 - continuant
BFO:0000001 - entity


In [27]:
print("Immediate parents:")
for parent in term.superclasses(distance=1):
    print(parent.id, "-", parent.name)

Immediate parents:
MONDO:0005314 - relapsing-remitting multiple sclerosis
MONDO:0005301 - multiple sclerosis


### Load embeddings

In [33]:
# Define the directory where your files are stored
load_snomed = False
if load_snomed:
    directory_path = "./data/snomed/embeddings"
    batch_name_prefix = "disorder_substance_finding_emb"
else:
    directory_path = "./data/mondo/embeddings"
    batch_name_prefix = "MONDO_emb"

# List all files in the directory that match the pattern
files = [f for f in os.listdir(directory_path) if f.startswith(f'{batch_name_prefix}_batch_') and f.endswith('.npy')]
print(files)
# Sort files to maintain the order, especially important if the batch index is used in processing
files.sort()

# Initialize an empty list to hold the data from each file
all_data = []

# Load each file and append the data to the list
for file in files:
    file_path = os.path.join(directory_path, file)
    data = np.load(file_path)
    all_data.append(data)

if load_snomed:
    # Concatenate all the arrays from the list into one
    all_reps_emb_full_snomed = np.concatenate(all_data, axis=0)
else:
    all_reps_emb_full = np.concatenate(all_data, axis=0)

['MONDO_emb_batch_1.npy', 'MONDO_emb_batch_0.npy']


In [35]:
all_reps_emb_full.shape

(129189, 768)

In [37]:
all_reps_emb_full_snomed.shape

NameError: name 'all_reps_emb_full_snomed' is not defined

In [39]:
with open("data/snomed/disorder_substance_finding_snomed_sf_id_pairs.json", "r") as f:
    snomed_sf_id_pairs = json.load(f)

In [41]:
with open("data/mondo/mondo_term_id_pairs.json", "r") as f:
    term_id_pairs = json.load(f)

In [43]:
with open("data/mondo/mondo_id_to_term_map.json", "r") as f:
    canonical_term_map = json.load(f)

# Test mappings

In [45]:
from neural_based_nen import map_query_to_terminology

In [46]:
tokenizer = AutoTokenizer.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")
model = AutoModel.from_pretrained("cambridgeltl/SapBERT-from-PubMedBERT-fulltext")

In [48]:
len(all_reps_emb_full), len(term_id_pairs)

(129189, 129189)

In [199]:
# Example usage
dist_threshold = 15
map_to_snomed = False
if map_to_snomed:
    embeddings_to_use = all_reps_emb_full_snomed
    corresponding_term_id = snomed_sf_id_pairs
else:
    embeddings_to_use = all_reps_emb_full
    corresponding_term_id = term_id_pairs
    
query = "pig inclusion conjunctivitis"
predicted_id, predicted_label, canonical_form, nearest_3_entities, nn_distance = map_query_to_terminology(query, tokenizer, model, embeddings_to_use, corresponding_term_id, canonical_term_map, dist_threshold=10)
print("Predicted ID:", predicted_id)
print("Predicted label:", predicted_label)
print("Distance:", nn_distance)
print("Canonical form:", canonical_form)
print("Nearest 3: ", nearest_3_entities)

Predicted ID: MONDO:0005808
Predicted label: inclusion conjunctivitis
Distance: 7.9029
Canonical form: inclusion conjunctivitis
Nearest 3:  [['inclusion conjunctivitis', 'MONDO:0005808'], ['adult inclusion conjunctivitis', 'MONDO:0005808'], ['pseudomembranous conjunctivitis', 'MONDO:0001217'], ['giant papillary conjunctivitis', 'MONDO:0002308'], ['pig disease, blue-eared', 'MONDO:0025494']]


In [193]:
canonical_term_map['MONDO:0016011']

'fetal alcohol syndrome'

In [195]:
canonical_term_map['MONDO:0000408']

'fetal alcohol spectrum disorder'